# ScriptSync — GPU Backend Server (runs inside Colab)
This turns this notebook into a live API server, using Colab's free GPU, that your Render-hosted website can call instead of doing transcription on slow free-tier CPU.

**Before running:** `Runtime > Change runtime type > T4 GPU`

**Steps:** Run cells 1 -> 2 -> 3 -> 4 in order. Cell 4 will print a public URL — copy that URL into Render's `COLAB_BACKEND_URL` environment variable.

**Important:** this notebook must stay open and running for the site to work. If it disconnects (Colab free tier times out after ~90 min idle, ~12h max), you'll need to re-run cell 4 and update the URL in Render again — it changes every time.

In [3]:
# Cell 1: Install dependencies
!pip install faster-whisper deep-translator moviepy flask flask-cors pyngrok -q

In [4]:
# Cell 2: Set your ngrok auth token
# Free account: sign up at https://dashboard.ngrok.com/signup
# Then copy your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
from pyngrok import ngrok

NGROK_AUTH_TOKEN = input('Paste your ngrok authtoken here: ').strip()
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
print('ngrok configured.')

Paste your ngrok authtoken here: 3Gdaz0XVDxzebCBuCFxdco6QBkQ_4Js6CzZnoHx56kQuVSE34
ngrok configured.


In [5]:
# Cell 3: Define the Flask API (mirrors ScriptSync's Render backend, but GPU-accelerated)
import os
import shutil
import tempfile
import torch

from flask import Flask, request, jsonify
from flask_cors import CORS
from faster_whisper import WhisperModel
from deep_translator import GoogleTranslator
import moviepy.editor as mp

app = Flask(__name__)
CORS(app)  # allow your Render-hosted frontend to call this server from the browser

app.config['MAX_CONTENT_LENGTH'] = 500 * 1024 * 1024  # 500MB

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
COMPUTE_TYPE = 'float16' if DEVICE == 'cuda' else 'int8'
# 'medium' is noticeably more accurate than 'small', and Colab's free T4 GPU
# handles it fine. Bump to 'large-v3' for the best accuracy if you don't mind
# it running a bit slower (still much faster than Render's CPU ever was).
MODEL_SIZE = 'medium'

print(f'Loading Whisper model "{MODEL_SIZE}" on {DEVICE}...')
model = WhisperModel(MODEL_SIZE, device=DEVICE, compute_type=COMPUTE_TYPE)
print('Model loaded.')

LANGUAGES = {
    'Hindi': 'hi', 'English': 'en', 'Marathi': 'mr', 'Bengali': 'bn',
    'Tamil': 'ta', 'Telugu': 'te', 'Gujarati': 'gu', 'Kannada': 'kn',
    'Malayalam': 'ml', 'Punjabi': 'pa', 'Urdu': 'ur',
    'Spanish': 'es', 'French': 'fr', 'German': 'de', 'Portuguese': 'pt',
    'Russian': 'ru', 'Chinese (Simplified)': 'zh-CN', 'Japanese': 'ja',
    'Korean': 'ko', 'Arabic': 'ar',
}


@app.route('/health')
def health():
    return jsonify({'status': 'ok', 'device': DEVICE, 'model': MODEL_SIZE})


@app.route('/api/languages')
def api_languages():
    return jsonify(LANGUAGES)


@app.route('/api/process', methods=['POST'])
def api_process():
    if 'video' not in request.files:
        return jsonify({'error': 'No video file received.'}), 400

    video = request.files['video']
    target_lang = request.form.get('language', 'hi')

    if video.filename == '':
        return jsonify({'error': 'Empty filename.'}), 400

    tmp_dir = tempfile.mkdtemp()
    try:
        video_path = os.path.join(tmp_dir, video.filename)
        video.save(video_path)

        audio_path = os.path.join(tmp_dir, 'audio.wav')
        clip = mp.VideoFileClip(video_path)
        clip.audio.write_audiofile(audio_path, logger=None)
        clip.close()

        segments_gen, info = model.transcribe(audio_path, beam_size=5, vad_filter=True)
        raw_segments = [
            {'start': round(seg.start, 2), 'end': round(seg.end, 2), 'original': seg.text.strip()}
            for seg in segments_gen if seg.text.strip()
        ]

        # Batched translation — fewer network calls than one-per-segment
        translator = GoogleTranslator(source='auto', target=target_lang)
        DELIM = '\n|||\n'
        BATCH_CHAR_LIMIT = 4000

        batches = []
        current_batch, current_len = [], 0
        for seg in raw_segments:
            seg_len = len(seg['original']) + len(DELIM)
            if current_batch and current_len + seg_len > BATCH_CHAR_LIMIT:
                batches.append(current_batch)
                current_batch, current_len = [], 0
            current_batch.append(seg)
            current_len += seg_len
        if current_batch:
            batches.append(current_batch)

        segments = []
        for batch in batches:
            joined = DELIM.join(s['original'] for s in batch)
            try:
                translated_joined = translator.translate(joined)
                parts = translated_joined.split(DELIM.strip())
                if len(parts) != len(batch):
                    parts = []
                    for s in batch:
                        try:
                            parts.append(translator.translate(s['original']))
                        except Exception:
                            parts.append(s['original'])
            except Exception:
                parts = [s['original'] for s in batch]

            for seg, translated in zip(batch, parts):
                segments.append({
                    'start': seg['start'],
                    'end': seg['end'],
                    'original': seg['original'],
                    'translated': translated.strip(),
                })

        return jsonify({
            'detected_language': info.language,
            'target_language': target_lang,
            'segments': segments,
        })

    except Exception as e:
        return jsonify({'error': str(e)}), 500
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)


print('Flask app defined. Run the next cell to start the server.')

Loading Whisper model "medium" on cuda...
Model loaded.
Flask app defined. Run the next cell to start the server.


In [ ]:
# Cell 4: Start the server + open the public tunnel
# This cell will keep running (that's expected) — it's your live server.
# Copy the printed URL into Render's COLAB_BACKEND_URL environment variable.
from pyngrok import ngrok

ngrok.kill()  # clear any old tunnels from a previous run
public_url = ngrok.connect(5000, bind_tls=True)
print('=' * 60)
print(f'Your Colab backend is live at: {public_url}')
print('Copy this URL into Render -> Environment -> COLAB_BACKEND_URL')
print('This URL changes every time you re-run this cell.')
print('=' * 60)

app.run(port=5000)

Your Colab backend is live at: NgrokTunnel: "https://popsicle-resemble-similarly.ngrok-free.dev" -> "http://localhost:5000"
Copy this URL into Render -> Environment -> COLAB_BACKEND_URL
This URL changes every time you re-run this cell.
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [18/Jul/2026 06:41:14] "POST /api/process HTTP/1.1" 200 -
